# 04 Replenishment and Warehouse Allocation

## 4.1 Business Objective

This notebook converts SKU classification results into replenishment and warehouse allocation recommendations.

The goal is to answer three management questions:

1. Which SKUs should be replenished first?
2. Which SKUs may create overstock risk?
3. Which SKUs should be prioritized for local warehouse inventory versus external or limited-stock strategies?

## 4.2 Data Foundation from Previous Notebooks

This notebook uses `sku_profile_classification.csv`, which was generated in `03_sku_classification.ipynb`.

The SKU classification output is based on cleaned product sales data from `01_data_cleaning.ipynb` and SQL-based business summaries from `02_sql_business_queries.ipynb`.

The data foundation excludes cancellations, returns, invalid sales records, duplicate rows, and non-product transaction lines.

This ensures that replenishment and warehouse allocation logic is built on valid physical product sales records.

In [43]:
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

outputs_dir = project_root / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

sku_profile = pd.read_csv(outputs_dir / "sku_profile_classification.csv")

print("SKU profile shape:", sku_profile.shape)
sku_profile.head()

SKU profile shape: (3917, 12)


,stock_code,total_units,total_revenue,active_months,avg_monthly_units,std_monthly_units,avg_unit_price,total_orders,description,demand_cv,sku_class,recommended_action
0,10002,860,759.89,5,172.000000,132.183584,1.045843,71,INFLATABLE POLITICAL GLOBE,0.768509,Regular,Maintain standard replenishment review
1,10080,303,119.09,7,43.285714,33.119553,0.455714,22,GROOVY CACTUS INFLATABLE,0.765138,Regular,Maintain standard replenishment review
2,10120,192,40.32,10,19.200000,15.454593,0.210000,29,DOGGY RUBBER,0.804927,Regular,Maintain standard replenishment review
3,10123C,5,3.25,2,2.500000,2.121320,0.650000,3,HEARTS WRAPPING TAPE,0.848528,Long-Tail,Limit stock and avoid excessive local warehous...
4,10124A,16,6.72,4,4.000000,0.816497,0.420000,5,SPOTS ON RED BOOKCOVER TAPE,0.204124,Long-Tail,Limit stock and avoid excessive local warehous...


## 4.3 Simulated Inventory Assumptions

The public sales dataset does not include actual inventory levels, supplier lead times, unit costs, storage volume, warehouse capacity, or fulfillment method.

To demonstrate how sales data can support replenishment planning and warehouse allocation, this notebook simulates inventory-related fields:

1. Current inventory.
2. Supplier lead time.
3. Storage volume per unit.
4. Unit cost.

These fields are simulated for portfolio demonstration purposes only. They do not represent actual company inventory data or real commercial decisions.

The purpose of this step is to show the analytical workflow: how cleaned sales data and SKU classification can be extended into inventory risk assessment and warehouse strategy recommendations.

In [44]:
np.random.seed(42)

sku_profile["current_inventory"] = np.where(
    sku_profile["sku_class"].isin(["High-Revenue Priority", "High-Turnover Stable"]),
    np.random.randint(20, 300, size=len(sku_profile)),
    np.random.randint(0, 120, size=len(sku_profile))
)

sku_profile["lead_time_days"] = np.random.choice(
    [7, 14, 21, 30, 45],
    size=len(sku_profile),
    p=[0.20, 0.35, 0.25, 0.15, 0.05]
)

sku_profile["storage_volume_per_unit"] = np.random.uniform(
    0.1, 3.0,
    size=len(sku_profile)
).round(2)

sku_profile["unit_cost"] = (sku_profile["avg_unit_price"] * np.random.uniform(0.35, 0.65, size=len(sku_profile))).round(2)

sku_profile.head()

,stock_code,total_units,total_revenue,active_months,avg_monthly_units,std_monthly_units,avg_unit_price,total_orders,description,demand_cv,sku_class,recommended_action,current_inventory,lead_time_days,storage_volume_per_unit,unit_cost
0,10002,860,759.89,5,172.000000,132.183584,1.045843,71,INFLATABLE POLITICAL GLOBE,0.768509,Regular,Maintain standard replenishment review,58,14,1.38,0.62
1,10080,303,119.09,7,43.285714,33.119553,0.455714,22,GROOVY CACTUS INFLATABLE,0.765138,Regular,Maintain standard replenishment review,75,30,0.31,0.27
2,10120,192,40.32,10,19.200000,15.454593,0.210000,29,DOGGY RUBBER,0.804927,Regular,Maintain standard replenishment review,48,14,1.23,0.09
3,10123C,5,3.25,2,2.500000,2.121320,0.650000,3,HEARTS WRAPPING TAPE,0.848528,Long-Tail,Limit stock and avoid excessive local warehous...,58,30,2.25,0.32
4,10124A,16,6.72,4,4.000000,0.816497,0.420000,5,SPOTS ON RED BOOKCOVER TAPE,0.204124,Long-Tail,Limit stock and avoid excessive local warehous...,30,7,1.22,0.26


## 4.4 Reorder Point and Safety Stock

This step calculates basic replenishment indicators.

The reorder point is calculated using average daily demand, supplier lead time, and safety stock.

Safety stock is estimated based on demand volatility. SKUs with higher demand volatility receive higher safety stock.

This logic helps identify SKUs that may need replenishment before inventory falls below the required level.

In [45]:
# Convert monthly demand to daily demand
sku_profile["avg_daily_demand"] = sku_profile["avg_monthly_units"] / 30

# Safety stock logic based on demand volatility
sku_profile["safety_stock"] = (
    sku_profile["avg_daily_demand"] *
    sku_profile["lead_time_days"] *
    (0.25 + sku_profile["demand_cv"].clip(0, 2) * 0.25)
).round(0)

# Reorder point
sku_profile["reorder_point"] = (
    sku_profile["avg_daily_demand"] * sku_profile["lead_time_days"] +
    sku_profile["safety_stock"]
).round(0)

# Recommended replenishment quantity
sku_profile["recommended_replenishment_qty"] = (
    sku_profile["reorder_point"] - sku_profile["current_inventory"]
).clip(lower=0).round(0)

sku_profile[
    [
        "stock_code",
        "description",
        "sku_class",
        "avg_monthly_units",
        "current_inventory",
        "lead_time_days",
        "safety_stock",
        "reorder_point",
        "recommended_replenishment_qty"
    ]
].head()

,stock_code,description,sku_class,avg_monthly_units,current_inventory,lead_time_days,safety_stock,reorder_point,recommended_replenishment_qty
0,10002,INFLATABLE POLITICAL GLOBE,Regular,172.000000,58,14,35.0,115.0,57.0
1,10080,GROOVY CACTUS INFLATABLE,Regular,43.285714,75,30,19.0,62.0,0.0
2,10120,DOGGY RUBBER,Regular,19.200000,48,14,4.0,13.0,0.0
3,10123C,HEARTS WRAPPING TAPE,Long-Tail,2.500000,58,30,1.0,4.0,0.0
4,10124A,SPOTS ON RED BOOKCOVER TAPE,Long-Tail,4.000000,30,7,0.0,1.0,0.0


## 4.5 Stockout and Overstock Risk

This step creates inventory risk flags:

1. Stockout Risk: current inventory is below the reorder point.
2. Overstock Risk: inventory coverage days are high while demand is low.
3. Normal: inventory position is within a reasonable range.

These flags are designed to support quick management review and help identify which SKUs need replenishment or inventory reduction attention.

In [46]:
sku_profile["inventory_coverage_days"] = np.where(
    sku_profile["avg_daily_demand"] > 0,
    sku_profile["current_inventory"] / sku_profile["avg_daily_demand"],
    np.inf
)

def assign_inventory_risk(row):
    if row["current_inventory"] < row["reorder_point"]:
        return "Stockout Risk"
    elif row["inventory_coverage_days"] > 180 and row["sku_class"] in ["Long-Tail", "Regular"]:
        return "Overstock Risk"
    else:
        return "Normal"

sku_profile["inventory_risk"] = sku_profile.apply(assign_inventory_risk, axis=1)

sku_profile[
    [
        "stock_code",
        "description",
        "sku_class",
        "current_inventory",
        "reorder_point",
        "inventory_coverage_days",
        "inventory_risk"
    ]
].head(20)

,stock_code,description,sku_class,current_inventory,reorder_point,inventory_coverage_days,inventory_risk
0,10002,INFLATABLE POLITICAL GLOBE,Regular,58,115.0,10.116279,Stockout Risk
1,10080,GROOVY CACTUS INFLATABLE,Regular,75,62.0,51.980198,Normal
2,10120,DOGGY RUBBER,Regular,48,13.0,75.000000,Normal
3,10123C,HEARTS WRAPPING TAPE,Long-Tail,58,4.0,696.000000,Overstock Risk
4,10124A,SPOTS ON RED BOOKCOVER TAPE,Long-Tail,30,1.0,225.000000,Overstock Risk
5,10124G,ARMY CAMO BOOKCOVER TAPE,Long-Tail,92,4.0,649.411765,Overstock Risk
6,10125,MINI FUNKY DESIGN TAPES,Regular,68,152.0,18.903475,Stockout Risk
7,10133,COLOURING PENCILS BROWN TUBE,High-Turnover Stable,141,101.0,14.810924,Normal
8,10135,COLOURING PENCILS BROWN TUBE,High-Turnover Stable,234,386.0,40.942127,Stockout Risk
9,11001,ASSTD DESIGN RACING CAR PEN,Regular,88,88.0,21.250774,Normal


## 4.6 Warehouse Allocation Logic

Each SKU is mapped to a suggested warehouse or fulfillment strategy.

The logic is based on SKU class, demand stability, revenue contribution, and inventory risk.

Suggested strategies include:

1. Local Warehouse Priority for high-revenue SKUs.
2. Stable Local Warehouse Inventory for stable high-turnover SKUs.
3. Small-Batch Replenishment / Monitor Closely for volatile high-turnover SKUs.
4. External or Limited Stock Strategy for long-tail SKUs.
5. Overstock Review / Reduce Replenishment for SKUs with overstock risk.
6. Standard Replenishment Review for regular SKUs.

In [47]:
def assign_warehouse_strategy(row):
    if row["inventory_risk"] == "Overstock Risk":
        return "Overstock Review / Reduce Replenishment"
    elif row["sku_class"] == "High-Revenue Priority":
        return "Local Warehouse Priority"
    elif row["sku_class"] == "High-Turnover Stable":
        return "Stable Local Warehouse Inventory"
    elif row["sku_class"] == "High-Turnover Volatile":
        return "Small-Batch Replenishment / Monitor Closely"
    elif row["sku_class"] == "Long-Tail":
        return "External or Limited Stock Strategy"
    else:
        return "Standard Replenishment Review"

sku_profile["warehouse_strategy"] = sku_profile.apply(assign_warehouse_strategy, axis=1)

sku_profile[
    [
        "stock_code",
        "description",
        "sku_class",
        "inventory_risk",
        "warehouse_strategy"
    ]
].head(20)

,stock_code,description,sku_class,inventory_risk,warehouse_strategy
0,10002,INFLATABLE POLITICAL GLOBE,Regular,Stockout Risk,Standard Replenishment Review
1,10080,GROOVY CACTUS INFLATABLE,Regular,Normal,Standard Replenishment Review
2,10120,DOGGY RUBBER,Regular,Normal,Standard Replenishment Review
3,10123C,HEARTS WRAPPING TAPE,Long-Tail,Overstock Risk,Overstock Review / Reduce Replenishment
4,10124A,SPOTS ON RED BOOKCOVER TAPE,Long-Tail,Overstock Risk,Overstock Review / Reduce Replenishment
5,10124G,ARMY CAMO BOOKCOVER TAPE,Long-Tail,Overstock Risk,Overstock Review / Reduce Replenishment
6,10125,MINI FUNKY DESIGN TAPES,Regular,Stockout Risk,Standard Replenishment Review
7,10133,COLOURING PENCILS BROWN TUBE,High-Turnover Stable,Normal,Stable Local Warehouse Inventory
8,10135,COLOURING PENCILS BROWN TUBE,High-Turnover Stable,Stockout Risk,Stable Local Warehouse Inventory
9,11001,ASSTD DESIGN RACING CAR PEN,Regular,Normal,Standard Replenishment Review


## 4.7 Management Output Tables

This step creates three management-facing output tables:

1. `replenishment_recommendations.csv`: SKUs that may require replenishment.
2. `overstock_risk_list.csv`: SKUs that may require overstock review or reduced replenishment.
3. `warehouse_allocation_summary.csv`: summary of SKU groups and recommended warehouse strategies.

These outputs translate the model results into practical tables that can support inventory review and warehouse planning.

In [48]:
replenishment_recommendations = (
    sku_profile[
        sku_profile["recommended_replenishment_qty"] > 0
    ]
    .sort_values(["sku_class", "recommended_replenishment_qty"], ascending=[True, False])
    [
        [
            "stock_code",
            "description",
            "sku_class",
            "avg_monthly_units",
            "current_inventory",
            "lead_time_days",
            "safety_stock",
            "reorder_point",
            "recommended_replenishment_qty",
            "inventory_risk",
            "warehouse_strategy"
        ]
    ]
)

overstock_risk_list = (
    sku_profile[
        sku_profile["inventory_risk"] == "Overstock Risk"
    ]
    .sort_values("inventory_coverage_days", ascending=False)
    [
        [
            "stock_code",
            "description",
            "sku_class",
            "avg_monthly_units",
            "current_inventory",
            "inventory_coverage_days",
            "inventory_risk",
            "warehouse_strategy"
        ]
    ]
)

warehouse_allocation_summary = (
    sku_profile
    .groupby(["sku_class", "warehouse_strategy"], as_index=False)
    .agg(
        sku_count=("stock_code", "count"),
        total_revenue=("total_revenue", "sum"),
        total_units=("total_units", "sum"),
        avg_inventory_coverage_days=("inventory_coverage_days", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)

replenishment_recommendations.to_csv(outputs_dir / "replenishment_recommendations.csv", index=False)
overstock_risk_list.to_csv(outputs_dir / "overstock_risk_list.csv", index=False)
warehouse_allocation_summary.to_csv(outputs_dir / "warehouse_allocation_summary.csv", index=False)

print("Saved output files:")
print("- outputs/replenishment_recommendations.csv")
print("- outputs/overstock_risk_list.csv")
print("- outputs/warehouse_allocation_summary.csv")

Saved output files:
- outputs/replenishment_recommendations.csv
- outputs/overstock_risk_list.csv
- outputs/warehouse_allocation_summary.csv


## 4.8 Management Implication

The replenishment and warehouse allocation outputs translate SKU-level sales analysis into operational decision support.

Based on the simulated inventory assumptions, the model identifies 1,432 SKUs with stockout risk and 924 SKUs with overstock risk.

The key business implication is that inventory and warehouse resources should not be managed uniformly across all products.

High-revenue SKUs and stable high-turnover SKUs should receive priority local warehouse attention because stockouts may directly affect sales performance. Volatile high-turnover SKUs should be monitored more frequently and replenished in smaller batches. Long-tail or overstock-risk SKUs should be reviewed carefully before additional replenishment and may be better suited for limited-stock or external fulfillment strategies.

This workflow demonstrates how cleaned e-commerce sales data can be converted into SKU-level replenishment recommendations, inventory risk flags, and warehouse allocation strategies.

## 4.9 Management KPI Summary

This step creates a final KPI summary table for management review.

The purpose is to consolidate the key project results into one output file, including cleaned data volume, SKU classification counts, inventory risk counts, and warehouse strategy outputs.

This table will be used in the project README and management summary.

In [49]:
management_kpi_summary = pd.DataFrame({
    "metric": [
        "valid_product_sales_rows",
        "returns_cancellations_rows",
        "non_product_rows_excluded",
        "sku_master_count",
        "sku_profile_count",
        "high_revenue_priority_sku_count",
        "long_tail_sku_count",
        "stockout_risk_sku_count",
        "overstock_risk_sku_count",
        "replenishment_recommendation_count",
        "warehouse_strategy_group_count"
    ],
    "value": [
        522716,
        10587,
        2162,
        3917,
        len(sku_profile),
        (sku_profile["sku_class"] == "High-Revenue Priority").sum(),
        (sku_profile["sku_class"] == "Long-Tail").sum(),
        (sku_profile["inventory_risk"] == "Stockout Risk").sum(),
        (sku_profile["inventory_risk"] == "Overstock Risk").sum(),
        len(replenishment_recommendations),
        len(warehouse_allocation_summary)
    ]
})

management_kpi_summary.to_csv(outputs_dir / "management_kpi_summary.csv", index=False)

management_kpi_summary

,metric,value
0,valid_product_sales_rows,522716
1,returns_cancellations_rows,10587
2,non_product_rows_excluded,2162
3,sku_master_count,3917
4,sku_profile_count,3917
5,high_revenue_priority_sku_count,784
6,long_tail_sku_count,1172
7,stockout_risk_sku_count,1432
8,overstock_risk_sku_count,924
9,replenishment_recommendation_count,1432


## 4.10 Final Summary Code Cell

The final summary cell confirms the main output shapes and distribution of inventory risk and warehouse strategy recommendations.

In [52]:
print("===== 04 Replenishment and Warehouse Allocation Summary =====")
print("Replenishment recommendations shape:", replenishment_recommendations.shape)
print("Overstock risk list shape:", overstock_risk_list.shape)
print("Warehouse allocation summary shape:", warehouse_allocation_summary.shape)
print("Management KPI summary shape:", management_kpi_summary.shape)

print("\nInventory risk distribution:")
print(sku_profile["inventory_risk"].value_counts())

print("\nWarehouse strategy distribution:")
print(sku_profile["warehouse_strategy"].value_counts())

print("\nOutput files:")
print("- outputs/replenishment_recommendations.csv")
print("- outputs/overstock_risk_list.csv")
print("- outputs/warehouse_allocation_summary.csv")
print("- outputs/management_kpi_summary.csv")

===== 04 Replenishment and Warehouse Allocation Summary =====
Replenishment recommendations shape: (1432, 11)
Overstock risk list shape: (924, 8)
Warehouse allocation summary shape: (7, 6)
Management KPI summary shape: (11, 2)

Inventory risk distribution:
inventory_risk
Normal            1561
Stockout Risk     1432
Overstock Risk     924
Name: count, dtype: int64

Warehouse strategy distribution:
warehouse_strategy
Standard Replenishment Review                  1597
Overstock Review / Reduce Replenishment         924
Local Warehouse Priority                        784
External or Limited Stock Strategy              335
Stable Local Warehouse Inventory                208
Small-Batch Replenishment / Monitor Closely      69
Name: count, dtype: int64

Output files:
- outputs/replenishment_recommendations.csv
- outputs/overstock_risk_list.csv
- outputs/warehouse_allocation_summary.csv
- outputs/management_kpi_summary.csv
